# 05 UC3M-LP check (M1b-6)
Runs on Colab. UC3M-LP (Spanish plates, Zenodo 17152029) is an **EU test set only**: never train on it, never share converted copies. Plan: PLAN.md (M1b-6).

Inspect first, convert later: this notebook downloads the zip, lists what is inside, and counts odd characters in its text files. The conversion cell is added after the real format has been seen. Run the cells top to bottom. Each cell prints its elapsed seconds.
Cell A prints folder and file names, so it is tagged `remove_output`; its output is not saved in the committed notebook.

In [ ]:
import subprocess, time
t0 = time.perf_counter()
SCRIPT = '''
set -e
cd /content
if [ -d vehicle-plate-recognition ]; then git -C vehicle-plate-recognition pull --ff-only; else git clone https://github.com/medrees-1000/vehicle-plate-recognition.git; fi
pip install -q uv  # approved raw pip: only to bootstrap uv on Colab
uv pip install --system -q --no-deps -e /content/vehicle-plate-recognition
'''
subprocess.run(["bash", "-c", SCRIPT], check=True)
print(f"elapsed {time.perf_counter() - t0:.1f} s")

In [ ]:
import time
import zipfile
from collections import Counter
from pathlib import Path, PurePosixPath
from google.colab import drive
from alpr import uc3m

t0 = time.perf_counter()
# Parameters
LOCAL = Path("/content/UC3M-LP.zip")
DRIVE_ZIP = Path("/content/drive/MyDrive/alpr/archives/UC3M-LP.zip")
README_MAX = 20 * 1024   # README/.md/.txt text is printed only for files up to this many bytes
README_FILES = 5         # at most this many doc files are looked at
TEXT_EXT = (".txt", ".json", ".xml", ".csv", ".yaml", ".yml")  # files scanned by the odd-character count
TEXT_MAX = 1 << 20       # text files above this many bytes are skipped (and counted)

drive.mount("/content/drive")
print(f"elapsed {time.perf_counter() - t0:.1f} s")

In [ ]:
t0 = time.perf_counter()
!python -m alpr.uc3m download {LOCAL} {DRIVE_ZIP}
print(f"elapsed {time.perf_counter() - t0:.1f} s")

In [ ]:
# Cell A: zip listing (prints names, tagged remove_output). Opens the zip read-only, extracts nothing.
t0 = time.perf_counter()
ZIP = LOCAL if LOCAL.exists() else DRIVE_ZIP
with zipfile.ZipFile(ZIP) as z:
    infos = [i for i in z.infolist() if not i.is_dir()]
    print(f"{ZIP}: {len(infos)} files, {sum(i.file_size for i in infos) / 1e9:.2f} GB uncompressed")
    folder = lambda n: "/".join(PurePosixPath(n).parts[:-1][:2]) or "(root)"
    ext = lambda n: PurePosixPath(n).suffix.lower() or "(none)"
    print("files per folder (first two levels):", dict(sorted(Counter(folder(i.filename) for i in infos).items())))
    print("files per extension:", dict(sorted(Counter(ext(i.filename) for i in infos).items())))
    print("files per (folder, extension):", dict(sorted(Counter((folder(i.filename), ext(i.filename)) for i in infos).items())))

    def is_doc(name: str) -> bool:
        base = name.lower().rsplit("/", 1)[-1]
        return base.startswith("readme") or base.endswith(".md") or (base.endswith(".txt") and name.count("/") <= 1)

    docs = [i for i in infos if is_doc(i.filename)]
    print(f"\ndoc-like files: {len(docs)} (looking at {min(len(docs), README_FILES)})")
    for i in docs[:README_FILES]:
        if i.file_size <= README_MAX:
            print(f"--- {i.filename} ({i.file_size} bytes)")
            print(z.read(i).decode("utf-8", "replace"))
        else:
            print(f"--- {i.filename}: {i.file_size} bytes, over {README_MAX}, not printed")
print(f"elapsed {time.perf_counter() - t0:.1f} s")

In [ ]:
# Blurred-character count: provisional until the label format is known (which symbol marks a blurred character is a guess).
# Counts every character that is not whitespace, a digit or A-Z, over the text-like members. No plate text, no names are printed.
t0 = time.perf_counter()
ZIP = LOCAL if LOCAL.exists() else DRIVE_ZIP
chars, files_with = Counter(), Counter()
with zipfile.ZipFile(ZIP) as z:
    members = [i for i in z.infolist() if not i.is_dir() and i.filename.lower().endswith(TEXT_EXT)]
    texts = [i for i in members if i.file_size <= TEXT_MAX]
    for i in texts:
        odd = Counter(c for c in z.read(i).decode("utf-8", "replace") if not (c.isspace() or "0" <= c <= "9" or "A" <= c <= "Z"))
        chars.update(odd)
        files_with.update(odd.keys())
print(f"text-like members: {len(members)}, scanned: {len(texts)}, skipped as too big: {len(members) - len(texts)}")
for c, n in chars.most_common(30):
    print(f"{c!r}: {n} times in {files_with[c]} files")
print(f"elapsed {time.perf_counter() - t0:.1f} s")